# Advanced Deep Learning – Lab 3.6

## Imports & Setup

In [1]:
import os, requests, zipfile, numpy as np, pandas as pd, matplotlib.pyplot as plt
from collections import Counter
from typing import Set, Dict, Tuple, List
import torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from datasets import load_dataset
import re, string

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cuda


---
# Task 3.6.1 — Semantic Qualities in Word Embeddings

## (a) Load GloVe Embeddings

In [2]:
# Download GloVe embeddings
data_path = '/tmp/glove'
glove_file = os.path.join(data_path, 'glove.6B.50d.txt')

if os.path.exists(data_path):
    import shutil
    shutil.rmtree(data_path)
os.makedirs(data_path)

print("Downloading GloVe embeddings...")
!wget -nv -t 0 --show-progress -O {glove_file} 'https://cloud.tu-ilmenau.de/s/m558re2RpoW8X2s/download/glove.6B.50d.txt'
print("Download complete!")

/tmp/glove/glove.6B 100%[===================>] 163.41M  22.0MB/s    in 8.9s    
2026-06-02 08:16:33 URL:https://cloud.tu-ilmenau.de/public.php/dav/files/m558re2RpoW8X2s? [171350079/171350079] -> "/tmp/glove/glove.6B.50d.txt" [1]
Download complete!


In [3]:
def read_glove_vecs(glove_file):
    with open(glove_file, 'r', encoding='utf-8') as f:
        words = set()
        word_to_vec_map = {}
        for line in f:
            line = line.strip().split()
            curr_word = line[0]
            words.add(curr_word)
            word_to_vec_map[curr_word] = np.array(line[1:], dtype=np.float64)
    return words, word_to_vec_map

words, word_to_vec_map = read_glove_vecs(glove_file)
print(f"Number of words in GloVe vocabulary: {len(words)}")
print(f"Shape of embedding vector for 'computer': {word_to_vec_map['computer'].shape}")

Number of words in GloVe vocabulary: 400000
Shape of embedding vector for 'computer': (50,)


## (b) Cosine Similarity

In [4]:
def cosine_similarity(vec1, vec2):
    dot_product = np.dot(vec1, vec2)
    norm_vec1 = np.linalg.norm(vec1)
    norm_vec2 = np.linalg.norm(vec2)
    if norm_vec1 == 0 or norm_vec2 == 0:
        return 0.0
    return dot_product / (norm_vec1 * norm_vec2)

def get_embedding(word):
    if word in word_to_vec_map:
        return word_to_vec_map[word]
    return None

# Test with word pairs
word_pairs = [('apple', 'banana'), ('apple', 'computer'), ('banana', 'computer')]
print("Cosine Similarity between word pairs:")
for word1, word2 in word_pairs:
    vec1, vec2 = get_embedding(word1), get_embedding(word2)
    if vec1 is not None and vec2 is not None:
        sim = cosine_similarity(vec1, vec2)
        print(f"{word1:15} <-> {word2:15} : {sim:.6f}")

Cosine Similarity between word pairs:
apple           <-> banana          : 0.560793
apple           <-> computer        : 0.640284
banana          <-> computer        : 0.106896


## (c) Find Closest Words

In [5]:
def find_closest_words(word, n=5):
    if word not in word_to_vec_map:
        return []
    word_vec = get_embedding(word)
    similarities = []
    for vocab_word in words:
        if vocab_word != word:
            vocab_vec = get_embedding(vocab_word)
            sim = cosine_similarity(word_vec, vocab_vec)
            similarities.append((vocab_word, sim))
    similarities.sort(key=lambda x: x[1], reverse=True)
    return similarities[:n]

print("Top 5 closest words to 'computer':")
closest = find_closest_words('computer', 5)
for word, sim in closest:
    print(f"{word:20} : {sim:.6f}")

Top 5 closest words to 'computer':
computers            : 0.916504
software             : 0.881499
technology           : 0.852556
electronic           : 0.812587
internet             : 0.806046


## (d) Word Analogies

In [6]:
def find_analogy(word1, word2, word3, top_n=5):
    vec1, vec2, vec3 = get_embedding(word1), get_embedding(word2), get_embedding(word3)
    if vec1 is None or vec2 is None or vec3 is None:
        return []
    target_vec = vec2 - vec1 + vec3
    target_vec = target_vec / np.linalg.norm(target_vec)
    similarities = []
    for vocab_word in words:
        if vocab_word not in [word1, word2, word3]:
            vocab_vec = get_embedding(vocab_word)
            sim = cosine_similarity(target_vec, vocab_vec)
            similarities.append((vocab_word, sim))
    similarities.sort(key=lambda x: x[1], reverse=True)
    return similarities[:top_n]

# Test analogies
analogies = [('king', 'queen', 'man'), ('france', 'paris', 'germany'), ('bad', 'worse', 'good')]
for word1, word2, word3 in analogies:
    print(f"\nAnalogy: {word1} is to {word2} as {word3} is to ?")
    results = find_analogy(word1, word2, word3, top_n=3)
    for word, sim in results:
        print(f"  {word:20} : {sim:.6f}")


Analogy: king is to queen as man is to ?
  woman                : 0.890391
  girl                 : 0.845373
  her                  : 0.784583

Analogy: france is to paris as germany is to ?
  berlin               : 0.918149
  frankfurt            : 0.818410
  munich               : 0.812075

Analogy: bad is to worse as good is to ?
  better               : 0.897457
  certainly            : 0.865128
  definitely           : 0.864294


## (e) Find Odd One Out

In [7]:
def find_odd_one_out(word_list):
    if len(word_list) < 2:
        return None
    embeddings = {w: get_embedding(w) for w in word_list if w in word_to_vec_map}
    if len(embeddings) != len(word_list):
        return None
    odd_scores = {}
    for word in word_list:
        similarities = [cosine_similarity(embeddings[word], embeddings[other])
                       for other in word_list if word != other]
        odd_scores[word] = np.mean(similarities)
    return min(odd_scores, key=odd_scores.get), odd_scores

word_list = ['computer', 'laptop', 'car', 'software']
odd_one, scores = find_odd_one_out(word_list)
print(f"Finding odd one out in: {word_list}")
for word, score in scores.items():
    print(f"  {word:20} : {score:.6f}")
print(f"\nOdd one out: '{odd_one}'")

Finding odd one out in: ['computer', 'laptop', 'car', 'software']
  computer             : 0.722575
  laptop               : 0.651840
  car                  : 0.487641
  software             : 0.643660

Odd one out: 'car'


## (f) Age Semantic Quality

In [8]:
young_words = ['junior', 'baby', 'child', 'teen', 'boy', 'girl', 'young']
old_words = ['senior', 'elder', 'grandpa', 'man', 'woman', 'grandma', 'old']

young_vecs = [get_embedding(w) for w in young_words if w in word_to_vec_map]
old_vecs = [get_embedding(w) for w in old_words if w in word_to_vec_map]

young_avg = np.mean(young_vecs, axis=0)
old_avg = np.mean(old_vecs, axis=0)
age_vector = old_avg - young_avg
age_vector = age_vector / np.linalg.norm(age_vector)

test_words = ['girl', 'boy', 'stone', 'student', 'birthday']
print("Applying age vector to test words:")
for test_word in test_words:
    if test_word not in word_to_vec_map:
        continue
    test_vec = get_embedding(test_word)
    aged_vec = test_vec + age_vector
    aged_vec = aged_vec / np.linalg.norm(aged_vec)
    best_match = max([(w, cosine_similarity(aged_vec, get_embedding(w)))
                     for w in old_words if w in word_to_vec_map], key=lambda x: x[1])
    print(f"{test_word:15} aged → {best_match[0]:15} (sim: {best_match[1]:.4f})")

Applying age vector to test words:
girl            aged → woman           (sim: 0.9193)
boy             aged → man             (sim: 0.8812)
stone           aged → old             (sim: 0.4906)
student         aged → senior          (sim: 0.7265)
birthday        aged → man             (sim: 0.4307)


## (g) Gender Semantic Quality

In [9]:
male_words = ['king', 'prince', 'man', 'boy', 'father', 'brother', 'uncle', 'son']
female_words = ['queen', 'princess', 'woman', 'girl', 'mother', 'sister', 'aunt', 'daughter']

male_vecs = [get_embedding(w) for w in male_words if w in word_to_vec_map]
female_vecs = [get_embedding(w) for w in female_words if w in word_to_vec_map]

male_avg = np.mean(male_vecs, axis=0)
female_avg = np.mean(female_vecs, axis=0)
gender_vector = female_avg - male_avg
gender_vector = gender_vector / np.linalg.norm(gender_vector)

test_words_gender = ['actor', 'teacher', 'doctor', 'nurse', 'poet']
print("Applying gender vector (male → female) to test words:")
for test_word in test_words_gender:
    if test_word not in word_to_vec_map:
        continue
    test_vec = get_embedding(test_word)
    gendered_vec = test_vec + gender_vector
    gendered_vec = gendered_vec / np.linalg.norm(gendered_vec)
    best_match = max([(w, cosine_similarity(gendered_vec, get_embedding(w)))
                     for w in female_words if w in word_to_vec_map], key=lambda x: x[1])
    print(f"{test_word:15} +gender → {best_match[0]:15} (sim: {best_match[1]:.4f})")

Applying gender vector (male → female) to test words:
actor           +gender → girl            (sim: 0.6000)
teacher         +gender → woman           (sim: 0.7329)
doctor          +gender → woman           (sim: 0.7735)
nurse           +gender → woman           (sim: 0.7213)
poet            +gender → mother          (sim: 0.5603)


---
# Task 3.6.2 — Text Classification with GloVe Embeddings

## Load AG News Dataset

In [10]:
print("Loading AG News dataset...")
dataset = load_dataset("fancyzhx/ag_news")
temp = dataset["train"].train_test_split(test_size=0.2, seed=42)
dataset['val'] = temp['test']
dataset['train'] = temp['train']

for split in ['train', 'val', 'test']:
    print(f"  {split}: {len(dataset[split])} samples")

Loading AG News dataset...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


  train: 96000 samples
  val: 24000 samples
  test: 7600 samples


## Preprocessing and Vocabulary

In [11]:
def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'\S+@\S+', '', text)
    text = re.sub(r'[^a-z\s]', '', text)
    return ' '.join(text.split())

word_to_idx = {word: idx for idx, word in enumerate(sorted(words))}
UNK_IDX = len(word_to_idx)
PAD_IDX = len(word_to_idx) + 1
word_to_idx['<UNK>'] = UNK_IDX
word_to_idx['<PAD>'] = PAD_IDX

print(f"Total words in mapping: {len(word_to_idx)}")

def text_to_indices(text, max_len=300):
    text = preprocess_text(text)
    indices = [word_to_idx.get(w, UNK_IDX) for w in text.split()[:max_len]]
    while len(indices) < max_len:
        indices.append(PAD_IDX)
    return torch.tensor(indices[:max_len], dtype=torch.long)

Total words in mapping: 400002


## Create Embedding Matrix

In [12]:
print("Creating embedding matrix from GloVe...")
embedding_dim = 50
embedding_matrix = np.random.randn(len(word_to_idx), embedding_dim) * 0.01

num_found = 0
for word, idx in word_to_idx.items():
    if word in word_to_vec_map:
        embedding_matrix[idx] = word_to_vec_map[word]
        num_found += 1

print(f"Embedding matrix shape: {embedding_matrix.shape}")
print(f"Words found in GloVe: {num_found}/{len(word_to_idx)} ({100*num_found/len(word_to_idx):.1f}%)")

Creating embedding matrix from GloVe...
Embedding matrix shape: (400002, 50)
Words found in GloVe: 400000/400002 (100.0%)


## Prepare DataLoaders

In [13]:
MAX_SEQ_LEN = 300
train_indices = torch.stack([text_to_indices(text, MAX_SEQ_LEN) for text in dataset['train']['text']])
val_indices = torch.stack([text_to_indices(text, MAX_SEQ_LEN) for text in dataset['val']['text']])
test_indices = torch.stack([text_to_indices(text, MAX_SEQ_LEN) for text in dataset['test']['text']])

train_labels = torch.tensor(dataset['train']['label'], dtype=torch.long)
val_labels = torch.tensor(dataset['val']['label'], dtype=torch.long)
test_labels = torch.tensor(dataset['test']['label'], dtype=torch.long)

BATCH_SIZE = 256
train_loader = DataLoader(TensorDataset(train_indices, train_labels), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(TensorDataset(val_indices, val_labels), batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(TensorDataset(test_indices, test_labels), batch_size=BATCH_SIZE, shuffle=False)

## Model Definition

In [14]:
class TextClassifierGloVe(nn.Module):
    def __init__(self, embedding_matrix, hidden_size, num_classes, freeze_embeddings=False):
        super().__init__()
        vocab_size, embedding_dim = embedding_matrix.shape
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=PAD_IDX)
        self.embedding.weight = nn.Parameter(torch.from_numpy(embedding_matrix).float())
        if freeze_embeddings:
            self.embedding.weight.requires_grad = False
        self.lstm = nn.LSTM(input_size=embedding_dim, hidden_size=hidden_size,
                           num_layers=2, batch_first=True, dropout=0.3)
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        x = self.embedding(x)
        _, (h_n, _) = self.lstm(x)
        h_final = h_n[-1]
        logits = self.fc(h_final)
        return logits

model_frozen = TextClassifierGloVe(embedding_matrix, 128, 4, freeze_embeddings=True).to(device)
model_trainable = TextClassifierGloVe(embedding_matrix, 128, 4, freeze_embeddings=False).to(device)
print("Models created!")

Models created!


## Training

In [15]:
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    for inputs, targets in loader:
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad()
        logits = model(inputs)
        loss = criterion(logits, targets)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

def eval_epoch(model, loader, criterion, device):
    model.eval()
    total_loss, correct = 0, 0
    with torch.no_grad():
        for inputs, targets in loader:
            inputs, targets = inputs.to(device), targets.to(device)
            logits = model(inputs)
            loss = criterion(logits, targets)
            total_loss += loss.item()
            correct += (logits.argmax(dim=1) == targets).sum().item()
    return total_loss / len(loader), correct / len(loader.dataset)

EPOCHS = 10
criterion = nn.CrossEntropyLoss()

print("Training FROZEN embedding model...")
optimizer_frozen = optim.Adam(model_frozen.parameters(), lr=0.001)
for epoch in range(1, EPOCHS + 1):
    tr_loss = train_epoch(model_frozen, train_loader, criterion, optimizer_frozen, device)
    val_loss, val_acc = eval_epoch(model_frozen, val_loader, criterion, device)
    if epoch % 2 == 0:
        print(f"Epoch {epoch:2}/{EPOCHS} | Train Loss: {tr_loss:.4f} | Val Acc: {val_acc:.4f}")

print("\nTraining TRAINABLE embedding model...")
optimizer_trainable = optim.Adam(model_trainable.parameters(), lr=0.001)
for epoch in range(1, EPOCHS + 1):
    tr_loss = train_epoch(model_trainable, train_loader, criterion, optimizer_trainable, device)
    val_loss, val_acc = eval_epoch(model_trainable, val_loader, criterion, device)
    if epoch % 2 == 0:
        print(f"Epoch {epoch:2}/{EPOCHS} | Train Loss: {tr_loss:.4f} | Val Acc: {val_acc:.4f}")

# Test evaluation
test_loss_frozen, test_acc_frozen = eval_epoch(model_frozen, test_loader, criterion, device)
test_loss_trainable, test_acc_trainable = eval_epoch(model_trainable, test_loader, criterion, device)

print("\n" + "="*60)
print("TEST SET RESULTS")
print("="*60)
print(f"Frozen Embeddings:    Accuracy: {test_acc_frozen:.4f}")
print(f"Trainable Embeddings: Accuracy: {test_acc_trainable:.4f}")
if test_acc_trainable > test_acc_frozen:
    print(f"\nTrainable is BETTER by {(test_acc_trainable - test_acc_frozen)*100:.2f}%")
else:
    print(f"\nFrozen is BETTER by {(test_acc_frozen - test_acc_trainable)*100:.2f}%")

Training FROZEN embedding model...
Epoch  2/10 | Train Loss: 1.3865 | Val Acc: 0.2521
Epoch  4/10 | Train Loss: 1.3864 | Val Acc: 0.2521
Epoch  6/10 | Train Loss: 1.3864 | Val Acc: 0.2475
Epoch  8/10 | Train Loss: 1.3864 | Val Acc: 0.2521
Epoch 10/10 | Train Loss: 1.3864 | Val Acc: 0.2475

Training TRAINABLE embedding model...
Epoch  2/10 | Train Loss: 1.3865 | Val Acc: 0.2521
Epoch  4/10 | Train Loss: 1.3864 | Val Acc: 0.2521
Epoch  6/10 | Train Loss: 1.3864 | Val Acc: 0.2483
Epoch  8/10 | Train Loss: 1.3864 | Val Acc: 0.2475
Epoch 10/10 | Train Loss: 1.3863 | Val Acc: 0.2521

TEST SET RESULTS
Frozen Embeddings:    Accuracy: 0.2500
Trainable Embeddings: Accuracy: 0.2500

Frozen is BETTER by 0.00%


---
# Task 3.6.3 — IMDB Sentiment Classification with Bidirectional LSTM

## Load and Analyze IMDB Dataset

In [16]:
imdb_dataset = load_dataset("stanfordnlp/imdb")
print(f"Train: {len(imdb_dataset['train'])}, Test: {len(imdb_dataset['test'])}")

# Split train into train/val
temp_imdb = imdb_dataset['train'].train_test_split(test_size=0.2, seed=42)
imdb_dataset['val'] = temp_imdb['test']
imdb_dataset['train'] = temp_imdb['train']

print(f"\nUpdated sizes:")
print(f"Train: {len(imdb_dataset['train'])}, Val: {len(imdb_dataset['val'])}, Test: {len(imdb_dataset['test'])}")

# examples
for i, (text, label) in enumerate(zip(imdb_dataset['train']['text'][:5], imdb_dataset['train']['label'][:5])):
    label_str = 'POSITIVE' if label == 1 else 'NEGATIVE'
    print(f"\nExample {i+1} ({label_str}): {text[:100]}...")

README.md:   0%|          | 0.00/7.81k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

Train: 25000, Test: 25000

Updated sizes:
Train: 20000, Val: 5000, Test: 25000

Example 1 (POSITIVE): Stage adaptations often have a major fault. They often come out looking like a film camera was simpl...

Example 2 (POSITIVE): 'The Rookie' was a wonderful movie about the second chances life holds for us and also puts an emoti...

Example 3 (NEGATIVE): OK,but does that make this a good movie?well,not really,in my opinion.there isn't a whole lot to rec...

Example 4 (POSITIVE): Two years ago I watched "The Matador" in cinema and I loved everything about this movie. Obviously, ...

Example 5 (POSITIVE): Well, I'll be honest: It is not exactly a Sholay. But you cant get a Sholay every week. In fact, you...


## Convert to Indices and Create DataLoaders

In [17]:
imdb_train_indices = torch.stack([text_to_indices(text, 300) for text in imdb_dataset['train']['text']])
imdb_val_indices = torch.stack([text_to_indices(text, 300) for text in imdb_dataset['val']['text']])
imdb_test_indices = torch.stack([text_to_indices(text, 300) for text in imdb_dataset['test']['text']])

imdb_train_labels = torch.tensor(imdb_dataset['train']['label'], dtype=torch.long)
imdb_val_labels = torch.tensor(imdb_dataset['val']['label'], dtype=torch.long)
imdb_test_labels = torch.tensor(imdb_dataset['test']['label'], dtype=torch.long)

BATCH_SIZE_IMDB = 512
imdb_train_loader = DataLoader(TensorDataset(imdb_train_indices, imdb_train_labels), batch_size=BATCH_SIZE_IMDB, shuffle=True)
imdb_val_loader = DataLoader(TensorDataset(imdb_val_indices, imdb_val_labels), batch_size=BATCH_SIZE_IMDB, shuffle=False)
imdb_test_loader = DataLoader(TensorDataset(imdb_test_indices, imdb_test_labels), batch_size=BATCH_SIZE_IMDB, shuffle=False)

## Bidirectional LSTM Model

In [18]:
class IMDBSentimentClassifier(nn.Module):
    def __init__(self, embedding_matrix, hidden_size=64, dropout=0.4):
        super().__init__()
        vocab_size, embedding_dim = embedding_matrix.shape
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=PAD_IDX)
        self.embedding.weight = nn.Parameter(torch.from_numpy(embedding_matrix).float())
        self.embedding.weight.requires_grad = False
        self.bilstm = nn.LSTM(input_size=embedding_dim, hidden_size=hidden_size,
                             num_layers=2, batch_first=True, dropout=dropout, bidirectional=True)
        self.fc = nn.Linear(hidden_size * 2, 1)

    def forward(self, x):
        x = self.embedding(x)
        out, (h_n, _) = self.bilstm(x)
        h_forward = h_n[-2]
        h_backward = h_n[-1]
        h_combined = torch.cat([h_forward, h_backward], dim=1)
        logits = self.fc(h_combined)
        return logits.squeeze(1)

imdb_model = IMDBSentimentClassifier(embedding_matrix, hidden_size=64, dropout=0.4).to(device)
print("Bidirectional LSTM model created!")

Bidirectional LSTM model created!


## Train and Evaluate

In [19]:
def train_imdb(model, loader, criterion, optimizer, device):
    model.train()
    loss_sum = 0
    for inputs, targets in loader:
        inputs, targets = inputs.to(device), targets.to(device).float()
        optimizer.zero_grad()
        logits = model(inputs)
        loss = criterion(logits, targets)
        loss.backward()
        optimizer.step()
        loss_sum += loss.item()
    return loss_sum / len(loader)

def eval_imdb(model, loader, criterion, device):
    model.eval()
    loss_sum, correct = 0, 0
    with torch.no_grad():
        for inputs, targets in loader:
            inputs, targets = inputs.to(device), targets.to(device).float()
            logits = model(inputs)
            loss = criterion(logits, targets)
            loss_sum += loss.item()
            correct += ((logits > 0) == targets.bool()).sum().item()
    return loss_sum / len(loader), correct / len(loader.dataset)

EPOCHS_IMDB = 20
criterion_imdb = nn.BCEWithLogitsLoss()
optimizer_imdb = optim.Adam(imdb_model.parameters(), lr=1e-2)

print("Training IMDB Sentiment Classifier with Bidirectional LSTM...")
for epoch in range(1, EPOCHS_IMDB + 1):
    tr_loss = train_imdb(imdb_model, imdb_train_loader, criterion_imdb, optimizer_imdb, device)
    val_loss, val_acc = eval_imdb(imdb_model, imdb_val_loader, criterion_imdb, device)
    if epoch % 5 == 0:
        print(f"Epoch {epoch:2}/{EPOCHS_IMDB} | Train Loss: {tr_loss:.4f} | Val Acc: {val_acc:.4f}")

# Test evaluation
test_loss, test_acc = eval_imdb(imdb_model, imdb_test_loader, criterion_imdb, device)
print(f"\nTest Accuracy: {test_acc:.4f} ({test_acc*100:.2f}%)")

Training IMDB Sentiment Classifier with Bidirectional LSTM...
Epoch  5/20 | Train Loss: 0.4668 | Val Acc: 0.8128
Epoch 10/20 | Train Loss: 0.3253 | Val Acc: 0.8538
Epoch 15/20 | Train Loss: 0.2349 | Val Acc: 0.8402
Epoch 20/20 | Train Loss: 0.1567 | Val Acc: 0.8520

Test Accuracy: 0.8545 (85.45%)
